In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt


# Define physical constant 
q = 1.602192e-19                    # Elementary charge
epsilon0 = 8.854187817e-12          # Vaccum permittibity
nint = 1e10                         # Intrinsic carrier concentration
kB = 1.38065e-23                    # Boltzmann constant
T = 300.0                           # Temperature
VT = kB*T/q                         # Thermal Voltage  


# Define Simulation variable  
L = 400e-9                          # Silicon Length  
N = 400                             # Number of grid points
ijunction = 200                     # Index of the PN Junction
l0 = L/N                            # Grid Spacing  
epsilon_si = 11.7                   # Silicon permittivity

ND = 1e20                           # Donor concentration
ND = ND * 1e6                       # Convert to /m^3
phiD = VT*math.asinh(0.5*ND/nint)   # Built-in Potential on N-side

NA = 1e19                           # Donor concentration
NA = NA * 1e6                       # Convert to /m^3
phiA = -VT*math.asinh(0.5*NA/nint)  # Built-in Potential on P-side

In [ ]:
# Intialize spatial & doping vector
# Position vector 
x = np.arange(N+1) * L / N          

# Doping profile (ND on left, -NA on right)
dop = np.zeros((N+1, 1))            

dop[0:ijunction] = ND               
dop[ijunction:N+1] = -NA

In [ ]:
# Initial potential value using quasi-equilibrium condition
phi = np.zeros((N+1, 1))
for ii in range(0, N+1):
    phi[ii] = VT * math.asinh(0.5 * dop[ii] / nint)

# Initialize carrier densities (holes and electrons)
hole = np.zeros((N+1, 1))
elec = np.zeros((N+1, 1))

In [ ]:
# Newton-Raphson iteration loop
for inewton in range(1, 10):

    # System matrix A & RHS vector b
    A = np.zeros((3*(N+1), 3*(N+1)))
    b = np.zeros((3*(N+1), 1))

    for ii in range(1, N):
        # Discretized Poisson equation
        b[3*ii] = epsilon_si * (phi[ii+1] - phi[ii]) - epsilon_si * (phi[ii] - phi[ii-1])
        A[3*ii, 3*(ii-1)] = epsilon_si
        A[3*ii, 3*ii]     = -2 * epsilon_si
        A[3*ii, 3*(ii+1)] = epsilon_si

        # Hole concentration
        b[3*ii+1] = hole[ii]
        A[3*ii+1, 3*ii+1] = 1.0

        # Electron concentration 
        b[3*ii+2] = elec[ii]
        A[3*ii+2, 3*ii+2] = 1.0

        # Non-linear charge terms to Poisson equation
        b[3*ii] += q * (hole[ii] - elec[ii] + dop[ii]) / epsilon0 * l0**2
        A[3*ii, 3*ii+1] =  q / epsilon0 * l0**2  
        A[3*ii, 3*ii+2] = -q / epsilon0 * l0**2   

        # Update hole equation with Non-linear term
        b[3*ii+1] -= nint * math.exp(-phi[ii] / VT)
        A[3*ii+1, 3*ii] = nint / VT * math.exp(-phi[ii] / VT)

        # Update electron equation with Non-linear term
        b[3*ii+2] -= nint * math.exp(phi[ii] / VT)
        A[3*ii+2, 3*ii] = -nint / VT * math.exp(phi[ii] / VT)

  
    # Boundary conditions
    # Potential boundary
    b[0] = phi[0] - phiD
    A[0, 0] = 1.0
    b[3*N] = phi[N] - phiA
    A[3*N, 3*N] = 1.0

    # Hole boundary (left and right)
    b[1] = hole[0] - nint * math.exp(-phiD / VT)
    A[1, 1] = 1.0
    b[3*N+1] = hole[N] - nint * math.exp(-phiA / VT)
    A[3*N+1, 3*N+1] = 1.0

    # Electron boundary (left and right)
    b[2] = elec[0] - nint * math.exp(phiD / VT)
    A[2, 2] = 1.0
    b[3*N+2] = elec[N] - nint * math.exp(phiA / VT)
    A[3*N+2, 3*N+2] = 1.0

    
    # Linear system solving & Update Potential value
    update = np.linalg.solve(A, b)

    # Update variables
    phi  = phi  - update[range(0, 3*(N+1), 3)]
    hole = hole - update[range(1, 3*(N+1), 3)]
    elec = elec - update[range(2, 3*(N+1), 3)]

In [ ]:
plt.plot(x/1e-9,phi,'bo-')

plt.xlabel('Position (nm)')
plt.ylabel('Electrostatic potential (V)')
plt.ylim(-1.5,1.5)

plt.show()